In [1]:
import os, json, requests
import pandas as pd
import numpy as np
from urllib.parse import quote
import time, random

import re
from tqdm.auto import tqdm
import ast

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from requests.exceptions import HTTPError, RequestException

from dotenv import load_dotenv

In [2]:
load_dotenv()  # 기본: 현재 폴더의 .env 로드

S2_API_KEY = os.getenv("S2_API_KEY")

In [3]:
paper = pd.read_csv("../../data/papers/papers_node_paper.csv")

In [4]:
paper

,paperId,url,title,publication,year,referenceCount,citationCount,authors,abstract,categories
0,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,https://www.semanticscholar.org/paper/8e0be569...,Random Forests,Machine-mediated learning,2001.0,25,110290,"[{'authorId': '2423230', 'name': 'L. Breiman'}]",NaN,"['Mathematics', 'Computer Science']"
1,df2b0e26d0599ce3e70df8a9da02e51594e0e992,https://www.semanticscholar.org/paper/df2b0e26...,BERT: Pre-training of Deep Bidirectional Trans...,North American Chapter of the Association for ...,2019.0,63,108712,"[{'authorId': '39172707', 'name': 'Jacob Devli...",We introduce a new language representation mod...,['Computer Science']
2,eb42cf88027de515750f230b23b1a057dc782108,https://www.semanticscholar.org/paper/eb42cf88...,Very Deep Convolutional Networks for Large-Sca...,International Conference on Learning Represent...,2014.0,43,108514,"[{'authorId': '34838386', 'name': 'K. Simonyan...",In this work we investigate the effect of the ...,"['Computer Science', 'Engineering']"
3,ad4fd2c149f220a62441576af92a8a669fe81246,https://www.semanticscholar.org/paper/ad4fd2c1...,Scikit-learn: Machine Learning in Python,Journal of machine learning research,2011.0,17,85166,"[{'authorId': '2570016', 'name': 'Fabian Pedre...",Scikit-learn is a Python module integrating a ...,['Computer Science']
4,4f8d648c52edf74e41b0996128aa536e13cc7e82,https://www.semanticscholar.org/paper/4f8d648c...,Deep Learning,International Journal of Semantic Computing,2016.0,2,70418,"[{'authorId': '2343609447', 'name': 'Xingbang ...",NaN,['Computer Science']
...,...,...,...,...,...,...,...,...,...,...
2211,e66496ae422cfd135e66a78e425c0b3fe8d4be22,https://www.semanticscholar.org/paper/e66496ae...,Prevalence of Autism Spectrum Disorder Among C...,Morbidity and mortality weekly report. Surveil...,2018.0,36,3752,NaN,Problem/Condition Autism spectrum disorder (AS...,"['Psychology', 'Medicine']"
2212,ddeb34dc450a7cdc9fd11cdb896b266be9fa1f05,https://www.semanticscholar.org/paper/ddeb34dc...,Prevalence of Autism Spectrum Disorder Among C...,Morbidity and mortality weekly report. Surveil...,2020.0,41,2915,NaN,Problem/Condition Autism spectrum disorder (AS...,"['Medicine', 'Psychology']"
2213,d7ac65d335b5d847f4f5826313a8732bc7abc7a8,https://www.semanticscholar.org/paper/d7ac65d3...,Calibrate Before Use: Improving Few-Shot Perfo...,International Conference on Machine Learning,2021.0,35,1694,NaN,GPT-3 can perform numerous tasks when provided...,['Computer Science']
2214,9ef33af1b2ebda2f2edd6c1394f314d7ac2f00f2,https://www.semanticscholar.org/paper/9ef33af1...,TweetEval: Unified Benchmark and Comparative E...,Findings,2020.0,33,868,NaN,The experimental landscape in natural language...,['Computer Science']


## 생성

In [5]:
MODEL = "JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4"  ## "Qwen/Qwen3-30B-A3B-GPTQ-Int4"

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

llm = LLM(
    model=MODEL,
    trust_remote_code=True,
    max_model_len=3600,
    gpu_memory_utilization=0.9,
    tensor_parallel_size=1,
    enable_prefix_caching=True,
    enforce_eager=True,
)

sampling = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    max_tokens=512,
)

def extract_json(text: str):
    if text is None:
        return None
    t = text.strip()
    if not t:
        return None

    # <think>...</think> 제거
    t = re.sub(r"<think>[\s\S]*?</think>", "", t).strip()

    # 코드블록(JSON) 우선 시도
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", t, flags=re.IGNORECASE)
    if m:
        cand = m.group(1).strip()
        # (a) 그대로
        try:
            return json.loads(cand)
        except Exception:
            # (b) 흔한 누락 보정: 마지막 } 없으면 붙여보기
            c2 = cand
            if c2.startswith("{") and not c2.endswith("}"):
                try:
                    return json.loads(c2 + "}")
                except Exception:
                    pass

    # 텍스트에서 JSON 객체 구간 추출
    start = t.find("{")
    if start == -1:
        return None

    # 마지막 }가 있으면 그 구간으로 파싱, 없으면 끝까지를 후보로
    end = t.rfind("}")
    if end != -1 and end > start:
        cand = t[start:end+1].strip()
        try:
            return json.loads(cand)
        except Exception:
            # 마지막 }가 있었는데도 실패하면, 혹시 뒤에 군더더기/잘림 대비 보정도 시도
            pass
    else:
        cand = t[start:].strip()

    # (a) 그대로
    try:
        return json.loads(cand)
    except Exception:
        pass

    # (b) 마지막 } 누락 보정
    if cand.startswith("{") and not cand.endswith("}"):
        try:
            return json.loads(cand + "}")
        except Exception:
            pass

    # (c) 마지막 ]까진 있는데 }만 없는 형태 보정: ...]  -> ...]}
    if cand.startswith("{") and not cand.endswith("}") and cand.rstrip().endswith("]"):
        try:
            return json.loads(cand.rstrip() + "}")
        except Exception:
            pass

    return None


def build_chat_prompt(system: str, user: str) -> str:
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

INFO 01-29 18:00:23 [utils.py:263] non-default args: {'trust_remote_code': True, 'max_model_len': 3600, 'enable_prefix_caching': True, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 01-29 18:00:25 [model.py:530] Resolved architecture: Qwen3MoeForCausalLM
WARNING 01-29 18:00:25 [model.py:1817] Your device 'Tesla V100-SXM2-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 01-29 18:00:25 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 01-29 18:00:25 [model.py:1545] Using max model len 3600
INFO 01-29 18:00:27 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-29 18:00:27 [gptq.py:99] Currently, the 4-bit gptq_gemm kernel for GPTQ is buggy. Please switch to gptq_marlin or gptq_bitblas.


Parse safetensors files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO 01-29 18:00:35 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 01-29 18:00:35 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
WARNING 01-29 18:00:35 [vllm.py:665] Enforce eager set, overriding optimization level to -O0
INFO 01-29 18:00:35 [vllm.py:765] Cudagraph is disabled under eager mode
WARNING 01-29 18:00:38 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=500420) INFO 01-29 18:00:46 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', speculative_config=None, tokenizer='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torc

(EngineCore_DP0 pid=500420) /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=500420) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=500420)   warnings.warn(


(EngineCore_DP0 pid=500420) INFO 01-29 18:00:50 [cuda.py:351] Using TRITON_ATTN attention backend out of potential backends: ('TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:03<00:15,  3.93s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:04<00:06,  2.03s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:09<00:06,  3.11s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:13<00:03,  3.66s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:17<00:00,  3.95s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:17<00:00,  3.60s/it]
(EngineCore_DP0 pid=500420) 


(EngineCore_DP0 pid=500420) INFO 01-29 18:01:17 [default_loader.py:291] Loading weights took 18.10 seconds
(EngineCore_DP0 pid=500420) INFO 01-29 18:01:18 [gpu_model_runner.py:3905] Model loading took 15.59 GiB memory and 29.731450 seconds
(EngineCore_DP0 pid=500420) WARNING 01-29 18:01:18 [fused_moe.py:1090] Using default MoE config. Performance might be sub-optimal! Config file not found at /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/configs/E=128,N=768,device_name=Tesla_V100-SXM2-32GB,dtype=int4_w4a16.json
(EngineCore_DP0 pid=500420) INFO 01-29 18:01:36 [gpu_worker.py:358] Available KV cache memory: 11.53 GiB
(EngineCore_DP0 pid=500420) INFO 01-29 18:01:36 [kv_cache_utils.py:1305] GPU KV cache size: 125,888 tokens
(EngineCore_DP0 pid=500420) INFO 01-29 18:01:36 [kv_cache_utils.py:1310] Maximum concurrency for 3,600 tokens per request: 34.97x
(EngineCore_DP0 pid=500420) INFO 01-29 18:01:37 [core.py:273] init en

In [6]:
DESC_PROMPT = """You are an expert scientific editor.

Write a SINGLE-SENTENCE, one-line description of the paper based ONLY on the title, abstract, and categories.
Be precise and conservative. Do NOT add facts not supported by the input.

Return ONLY valid, raw JSON. No extra text. No markdown.

### Output schema (must follow exactly)
{
  "description": "..."
}

### Rules
- "description" MUST be one sentence in Korean.
- 18–30 words (prefer concise, information-dense).
- Mention the paper's main contribution or purpose (method + task/problem + key idea).
- You MAY use categories only as a high-level field hint (e.g., NLP, CV, medicine) but do NOT infer specific methods/results from them.
- Avoid hype and vague phrases (e.g., "novel", "state-of-the-art") unless explicitly stated.
- If abstract is missing/empty: rely only on the title + categories; be extra conservative and keep it very short (8–16 words).
- Do not include citations, author names, or URLs.

### Input
Title: "{INPUT_TITLE}"
Abstract: "{INPUT_ABSTRACT}"
Categories: "{INPUT_CATEGORIES}"

Output: """


def add_description_to_nodes(papers_df: pd.DataFrame) -> pd.DataFrame:
    df = papers_df.copy()

    df["title"] = df.get("title", "").fillna("").astype(str)
    df["abstract"] = df.get("abstract", "").fillna("").astype(str)
    if "categories" not in df.columns:
        df["categories"] = ""

    def _cats_to_str(cats) -> str:
        if cats is None:
            return ""
        if isinstance(cats, (list, tuple, set)):
            parts = []
            for x in cats:
                s = str(x).strip()
                if s:
                    parts.append(s)
            return "; ".join(parts)
        s = str(cats).strip()
        s = re.sub(r"^\[|\]$", "", s)
        s = s.replace("'", "").replace('"', "").strip()
        return s

    df["categories_str"] = df["categories"].apply(_cats_to_str)

    chat_inputs = []
    row_keys = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc="desc: build prompts"):
        title = row["title"].strip()
        abstract = row["abstract"].strip()
        cats_str = row["categories_str"].strip()

        system_msg = (DESC_PROMPT
                      .replace("{INPUT_TITLE}", title)
                      .replace("{INPUT_ABSTRACT}", abstract)
                      .replace("{INPUT_CATEGORIES}", cats_str))

        user_msg = 'Return ONLY raw JSON with key "description".'
        chat_inputs.append(build_chat_prompt(system_msg, user_msg))
        row_keys.append(i)

    generations = llm.generate(chat_inputs, sampling)

    desc_map = {}
    raw_map = {}
    ok_map = {}

    def _clean_desc(s: str) -> str:
        if not isinstance(s, str):
            return ""
        t = s.strip()
        t = re.sub(r"\s+", " ", t).strip()
        if t and t[-1] not in ".?!":
            t = t + "."
        return t

    for idx, gen in tqdm(list(zip(row_keys, generations)), total=len(row_keys), desc="desc: parse outputs"):
        raw = gen.outputs[0].text.strip()
        parsed = extract_json(raw)

        ok = isinstance(parsed, dict) and isinstance(parsed.get("description"), str)

        raw_map[idx] = raw
        ok_map[idx] = ok

        if ok:
            desc_map[idx] = _clean_desc(parsed["description"])
        else:
            desc_map[idx] = ""

    df["description"] = df.index.map(lambda i: desc_map.get(i, ""))
    df["desc_raw"] = df.index.map(lambda i: raw_map.get(i, ""))
    df["desc_ok"] = df.index.map(lambda i: ok_map.get(i, False))

    df.drop(columns=["categories_str"], inplace=True, errors="ignore")

    return df

In [7]:
out_df = add_description_to_nodes(paper)
print(out_df["desc_ok"].value_counts(dropna=False))
out_df.loc[~out_df["desc_ok"], ["title", "description", "desc_raw"]].head(10)

desc: build prompts:   0%|          | 0/2216 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/2216 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2216 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

desc: parse outputs:   0%|          | 0/2216 [00:00<?, ?it/s]

desc_ok
True    2216
Name: count, dtype: int64


,title,description,desc_raw


In [12]:
out_df[['paperId','url','title','publication','year','referenceCount','citationCount','authors','abstract','categories','description']].to_csv("../../data/papers/papers_node_paper_with_desc.csv", index=False)

In [13]:
out_df

,paperId,url,title,publication,year,referenceCount,citationCount,authors,abstract,categories,description,desc_raw,desc_ok
0,8e0be569ea77b8cb29bb0e8b031887630fe7a96c,https://www.semanticscholar.org/paper/8e0be569...,Random Forests,Machine-mediated learning,2001.0,25,110290,"[{'authorId': '2423230', 'name': 'L. Breiman'}]",,"['Mathematics', 'Computer Science']","랜덤 포레스트는 분류 및 회귀 문제를 해결하기 위한 앙상블 학습 방법으로, 여러 개...","{\n ""description"": ""랜덤 포레스트는 분류 및 회귀 문제를 해결하기...",True
1,df2b0e26d0599ce3e70df8a9da02e51594e0e992,https://www.semanticscholar.org/paper/df2b0e26...,BERT: Pre-training of Deep Bidirectional Trans...,North American Chapter of the Association for ...,2019.0,63,108712,"[{'authorId': '39172707', 'name': 'Jacob Devli...",We introduce a new language representation mod...,['Computer Science'],BERT는 양방향 트랜스포머 기반의 깊은 표현을 사전 훈련하여 질문 응답 및 언어 ...,"{\n ""description"": ""BERT는 양방향 트랜스포머 기반의 깊은 표현...",True
2,eb42cf88027de515750f230b23b1a057dc782108,https://www.semanticscholar.org/paper/eb42cf88...,Very Deep Convolutional Networks for Large-Sca...,International Conference on Learning Represent...,2014.0,43,108514,"[{'authorId': '34838386', 'name': 'K. Simonyan...",In this work we investigate the effect of the ...,"['Computer Science', 'Engineering']",깊이가 16-19개의 가중치 계층에 이르는 매우 깊은 컨볼루션 네트워크를 사용하여 ...,"{\n ""description"": ""깊이가 16-19개의 가중치 계층에 이르는 매...",True
3,ad4fd2c149f220a62441576af92a8a669fe81246,https://www.semanticscholar.org/paper/ad4fd2c1...,Scikit-learn: Machine Learning in Python,Journal of machine learning research,2011.0,17,85166,"[{'authorId': '2570016', 'name': 'Fabian Pedre...",Scikit-learn is a Python module integrating a ...,['Computer Science'],스키킷-런은 중규모 지도 및 비지도 학습 문제에 적합한 다양한 최첨단 기계학습 알고...,"{\n ""description"": ""스키킷-런은 중규모 지도 및 비지도 학습 문제...",True
4,4f8d648c52edf74e41b0996128aa536e13cc7e82,https://www.semanticscholar.org/paper/4f8d648c...,Deep Learning,International Journal of Semantic Computing,2016.0,2,70418,"[{'authorId': '2343609447', 'name': 'Xingbang ...",,['Computer Science'],딥러닝은 컴퓨터 과학 분야에서 신경망 기반의 학습 방법을 연구하는 기술이다.,"{\n ""description"": ""딥러닝은 컴퓨터 과학 분야에서 신경망 기반의 ...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2211,e66496ae422cfd135e66a78e425c0b3fe8d4be22,https://www.semanticscholar.org/paper/e66496ae...,Prevalence of Autism Spectrum Disorder Among C...,Morbidity and mortality weekly report. Surveil...,2018.0,36,3752,NaN,Problem/Condition Autism spectrum disorder (AS...,"['Psychology', 'Medicine']",2014년도 자폐 스펙트럼 장애 모니터링 네트워크 데이터를 바탕으로 미국 11개 지...,"{\n ""description"": ""2014년도 자폐 스펙트럼 장애 모니터링 네트...",True
2212,ddeb34dc450a7cdc9fd11cdb896b266be9fa1f05,https://www.semanticscholar.org/paper/ddeb34dc...,Prevalence of Autism Spectrum Disorder Among C...,Morbidity and mortality weekly report. Surveil...,2020.0,41,2915,NaN,Problem/Condition Autism spectrum disorder (AS...,"['Medicine', 'Psychology']",2016년도 미국 11개 지역에서 8세 아동을 대상으로 자폐 스펙트럼 장애의 유병률...,"{\n ""description"": ""2016년도 미국 11개 지역에서 8세 아동을...",True
2213,d7ac65d335b5d847f4f5826313a8732bc7abc7a8,https://www.semanticscholar.org/paper/d7ac65d3...,Calibrate Before Use: Improving Few-Shot Perfo...,International Conference on Machine Learning,2021.0,35,1694,NaN,GPT-3 can perform numerous tasks when provided...,['Computer Science'],언어 모델의 예측 편향을 보정하여 희소 샘플 학습 성능을 안정화하고 정확도를 향상시...,"{\n ""description"": ""언어 모델의 예측 편향을 보정하여 희소 샘플 ...",True
2214,9ef33af1b2ebda2f2edd6c1394f314d7ac2f00f2,https://www.semanticscholar.org/paper/9ef33af1...,TweetEval: Unified Benchmark and Comparative E...,Findings,2020.0,33,868,NaN,The experimental landscape in natural language...,['Computer Science'],"트윗 분류를 위한 통합 벤치마크인 트윗평가(TweetEval)를 제안하고, 다양한 ...","{\n ""description"": ""트윗 분류를 위한 통합 벤치마크인 트윗평가(T...",True
